# K3: baselines (MostPop, DecayPop, iALS)

Generated by `scripts/make_notebooks.py`; see `docs/kaggle_runbook.md`. Run it headless:
**Save Version -> Save & Run All (Commit)**. No cell needs editing.

- **Accelerator:** None (CPU). **Internet:** ON.
- **Attach (Add Input):** the K2 output for SIZE
- **Expected time:** 50m < 0.5 h, 500m 1-3 h
- **Output (`/kaggle/working/out`):** `results/*.json` (val grid + test refits), marker `_HOOKREC_RESULTS_baselines`

In [ ]:
# Parameters: the defaults work unedited (attach inputs, then Save Version ->
# Save & Run All). Generated by scripts/make_notebooks.py.
REPO_URL = "https://github.com/Arubad/hookseq.git"  # the HookSeq repo (private: add a GITHUB_TOKEN secret)
GIT_REF = "main"  # branch or tag to run (pin a tag for reproducible runs)
SIZE = "50m"  # Yambda size: '50m' or '500m'
CONFIG = "configs/baselines.yaml"  # baseline grids
RUN_SPLITS = ["val", "test"]  # val grid search, then the test refit
MODELS = "mostpop,decaypop,ials"  # comma-separated
QUICK = False  # True = first grid value only (smoke test)

# fixed Kaggle paths
INPUT_ROOT = "/kaggle/input"
WORK = "/kaggle/working"
OUT = "/kaggle/working/out"
REPO = "/kaggle/working/repo"

import os
import time

# train.max_hours is measured from here (the session start), not from the script start
os.environ.setdefault("HOOKREC_SESSION_START", str(int(time.time())))
print("HOOKREC_SESSION_START =", os.environ["HOOKREC_SESSION_START"])

In [ ]:
# Clone the repo and install it. The image's torch is used (no torch install).
import importlib
import os
import shutil
import subprocess
import sys


def _stream(cmd):
    """Run cmd, streaming its output; raise on a non-zero exit (a `!cmd` magic would not)."""
    p = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in p.stdout:
        print(line, end="", flush=True)
    if p.wait() != 0:
        raise RuntimeError(f"{' '.join(cmd[:5])} ... exited {p.returncode}")


if os.path.exists(REPO):
    shutil.rmtree(REPO)
token = ""
try:  # optional, only for a private repo: Add-ons -> Secrets -> GITHUB_TOKEN (attached)
    from kaggle_secrets import UserSecretsClient

    token = UserSecretsClient().get_secret("GITHUB_TOKEN") or ""
except Exception:
    token = ""
url = REPO_URL
if token and REPO_URL.startswith("https://"):
    url = "https://x-access-token:" + token + "@" + REPO_URL[len("https://"):]
r = subprocess.run(
    ["git", "clone", "--depth", "1", "-b", GIT_REF, url, REPO], capture_output=True, text=True
)
if r.returncode != 0:
    log = r.stdout + r.stderr
    if token:
        log = log.replace(token, "***")
    raise RuntimeError(
        f"git clone {REPO_URL} -b {GIT_REF} failed (exit {r.returncode}):\n{log}\n"
        "Check REPO_URL / GIT_REF and that Internet is ON; a private repo needs the "
        "GITHUB_TOKEN secret attached to this notebook."
    )
# the clone is saved with the output: never leave the token in .git/config
subprocess.run(["git", "-C", REPO, "remote", "set-url", "origin", REPO_URL], check=True)
del token, url
_stream([sys.executable, "-m", "pip", "install", "-q", "-e", REPO])
sys.path.insert(0, os.path.join(REPO, "src"))
importlib.invalidate_caches()
SHA = subprocess.run(
    ["git", "-C", REPO, "rev-parse", "HEAD"], capture_output=True, text=True, check=True
).stdout.strip()
print(f"hookrec {REPO_URL} {GIT_REF} @ {SHA}")

In [ ]:
# Environment + input discovery (inputs are found by marker files; no slugs to edit).
import os
import sys
from pathlib import Path

from hookrec.utils.kaggle import (
    check_disk, env_report, find_inputs, find_one, find_processed, has_run, run_logged,
)

env_report(WORK)

PROCESSED = find_processed(SIZE, INPUT_ROOT)
print("processed:", PROCESSED)

In [ ]:
import sys
from pathlib import Path

RESULTS = Path(OUT, "results")
for split in RUN_SPLITS:  # test refits each model's best val config, so val runs first
    cmd = [
        sys.executable, f"{REPO}/scripts/run_baselines.py", "--processed", PROCESSED,
        "--config", os.path.join(REPO, CONFIG), "--split", split, "--out", RESULTS,
        "--models", MODELS,
    ]
    if QUICK:
        cmd.append("--quick")
    run_logged(cmd, cwd=REPO)
(RESULTS / "_HOOKREC_RESULTS_baselines").write_text("")

In [ ]:
from hookrec.eval.report import results_table
from hookrec.utils.results import load_runs

check_disk(WORK)
for p in sorted(RESULTS.glob("*.json")):
    print(p.name)
print(results_table(load_runs(RESULTS), split="test", size=SIZE))